# R5.1 (research track): independent scoring of the twins

R2's label renders each twin's held-out views at the method's **own** estimated poses. That needs no ground truth, but the R3b probe measures the same thing, so a reviewer can call the probe's success circular. This notebook re-scores every twin the way pose-free novel-view-synthesis papers do, against the **true** cameras:

| Score | How |
|---|---|
| `own_pose` | R2's protocol, recomputed: held-out views at the method's own poses (sanity check against R2's labels) |
| `aligned` | Sim(3) from the reference cameras onto the twin's **training** cameras; the held-out reference cameras are moved into the twin's frame and rendered with the reference intrinsics, against the real photo |
| `refined` | `aligned`, then each held-out pose refined by test-time optimisation (Gaussians frozen; Adam on L1, 200 iterations), as in NoPe-NeRF and CF-3DGS |

For every chunk (105, R2's) the official poses (`ref`) and every usable method get R2's exact 3DGS recipe (MCMC, 300k cap, 3k steps). Then `r5_eval.py` scores the checkpoint. The local analysis reruns R2–R4 with `refined` as the label and checks whether the conclusions hold.

**Input:** R2's reconstructions (the `r2-models` Dataset you made for R3b). **Output:** `r5/r5_scores.jsonl`.

**How to run (Kaggle):**
1. *Create → New Notebook → File → Import Notebook* → this file.
2. *Add Input* → *Datasets* → **r2-models** (uploaded for R3b).
3. *Settings*: **GPU T4 x2**, **Internet on**.
4. *Save Version → Save & Run All (Commit)*. About **13 GPU-hours**, so it stops itself at 11 h.
5. **Run 2:** *Add Input* → this notebook's Output (run 1), commit again; finished chunks are skipped. Then download the `r5` folder.

## 1. Settings

In [ ]:
import os, sys, glob, json, time, re, subprocess, shutil
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # Kaggle T4 x2: use one GPU (gsplat would otherwise go multi-GPU, F2.2)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                     capture_output=True, text=True).stdout)

ALL_SCENES = ['garden', 'bicycle', 'stump', 'bonsai', 'counter', 'kitchen', 'room']
SCENES     = ALL_SCENES
RES        = dict(garden='images_8', bicycle='images_8', stump='images_8',            # as in R2
                  bonsai='images_4', counter='images_4', kitchen='images_4', room='images_4')
METHODS    = ['ref', 'classical', 'vggt', 'vggt_ba']
STEPS      = 3_000                              # R2's label recipe
TTO_ITERS  = 200                                # test-time pose refinement per held-out view
CAP        = 300_000                            # same Gaussian cap as R2's labels
MIN_COVERAGE = 0.9                              # as R2: a method must register >= 90% of the chunk
BUDGET_H   = 11.0
T_START    = time.time()

PROJECT = '/kaggle/working'                  # becomes the Output
WORK    = '/kaggle/tmp'
OUT     = f'{PROJECT}/r5'
R5      = f'{WORK}/r5'                      # per-chunk, per-method scene folders (images are symlinks)
SCORES  = f'{OUT}/r5_scores.jsonl'
for d in [f'{PROJECT}/cache', OUT, f'{OUT}/logs', WORK, f'{WORK}/hybrid_sfm', R5]:
    os.makedirs(d, exist_ok=True)

# R2's reconstructions: the r2-models Dataset (Kaggle unpacks the zip; an unpacked copy is made if it didn't)
def find_models():
    for d in glob.glob('/kaggle/input/**/r2_models', recursive=True) + [f'{WORK}/r2_models']:
        if os.path.isdir(d) and os.path.isdir(f'{d}/garden_window24_0'):
            return d
    for z in glob.glob('/kaggle/input/**/r2_models.zip', recursive=True):
        import zipfile
        zipfile.ZipFile(z).extractall(WORK)
        return f'{WORK}/r2_models'
    raise SystemExit('❌ R2 models not found: Add Input → Datasets → r2-models (the uploaded r2_models.zip)')
MODELS = find_models()
print('R2 models:', MODELS, len(os.listdir(MODELS)), 'chunks')

# Scores from an earlier run (its Output added as an Input) are merged in and skipped
prev = [f for f in glob.glob('/kaggle/input/**/r5_scores.jsonl', recursive=True)]
if prev and not os.path.exists(SCORES):
    with open(SCORES, 'w') as o:
        for f in prev:
            o.write(open(f).read().rstrip('\n') + '\n')
    print('merged scores from', prev)

import socket
try:
    socket.create_connection(('pypi.org', 443), timeout=10).close(); print('internet: ok')
except OSError:
    raise SystemExit('❌ No internet. Kaggle: Settings → Internet → On (needs a phone-verified account). Then run again.')

## 2. Install gsplat

Same cells as R1–R3b.

In [ ]:
import importlib.util
MARKER = '/tmp/.gs_deps_ok'

if not os.path.exists(MARKER):
    gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gsplat==1.5.3'], check=True)

    REPO = f'{WORK}/gsplat'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', 'v1.5.3',
                        'https://github.com/nerfstudio-project/gsplat.git', REPO], check=True)
    reqs = open(f'{REPO}/examples/requirements.txt').read()

    # F0.1 + F0.2: git deps (several glued onto one line) -> split, build without isolation
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ninja', 'setuptools', 'wheel'], check=True)
    if 'T4' in gpu_name:
        os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'   # faster builds, T4 only
    for url in re.findall(r'git\+https://github\.com/[^@\s]+@[0-9a-f]{40}', reqs):
        print('building', url.split('/')[-1][:60])
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', url], check=True)

    # F0.3: drop git lines + numpy<2 pin, install the rest
    rest = [l for l in reqs.splitlines() if 'git+https' not in l and not l.lower().startswith('numpy')]
    open('/tmp/reqs.txt', 'w').write('\n'.join(rest))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/tmp/reqs.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'remotezip', 'plyfile'], check=True)

    # F0.5: pycolmap fork uses np.uint64(-1) -> OverflowError on numpy 2 (patch the file, don't import it)
    f = os.path.join(os.path.dirname(importlib.util.find_spec('pycolmap').origin), 'scene_manager.py')
    s = open(f).read()
    if 'np.uint64(-1)' in s:   # harmless on numpy 1, required on numpy 2
        open(f, 'w').write(s.replace('np.uint64(-1)', 'np.uint64(18446744073709551615)')); print('patched pycolmap')

    # F0.4: make gsplat's examples/datasets a regular package (beats HF `datasets`)
    open(f'{REPO}/examples/datasets/__init__.py', 'a').close()

    open(MARKER, 'w').write('ok')

REPO = f'{WORK}/gsplat'
check = subprocess.run([sys.executable, '-c',
    'import numpy, gsplat, fused_ssim, pycolmap, tyro; print("numpy", numpy.__version__, "| gsplat", gsplat.__version__)'],
    capture_output=True, text=True, cwd=f'{REPO}/examples')
print(check.stdout or check.stderr[-2000:])

In [ ]:
os.environ['GSPLAT_PREBUILT_DIR'] = '/kaggle/tmp/gsplat_prebuilt'
# F1.5 fix, Kaggle version: the compiled library is saved to the Output (cache/gsplat_jit/...).
# To skip the ~15 min compile next time: add that Output as a Kaggle Dataset and point JIT_CACHE at it.
import torch, importlib.util
PREBUILT = '/kaggle/tmp/gsplat_prebuilt'
cap = torch.cuda.get_device_capability()
KEY = f"gsplat1.5.3_torch{torch.__version__}_py{sys.version_info.major}{sys.version_info.minor}_sm{cap[0]}{cap[1]}"
JIT_CACHE = f'{PROJECT}/cache/gsplat_jit/{KEY}'

# 1. Patch gsplat's loader: use $GSPLAT_PREBUILT_DIR/gsplat_cuda.so if it exists, otherwise behave as before
BACKEND = os.path.join(os.path.dirname(importlib.util.find_spec('gsplat').origin), 'cuda', '_backend.py')
src = open(BACKEND).read()
if 'GSPLAT_PREBUILT_DIR' not in src:
    old = '    from gsplat import csrc as _C\n'
    new = ('    _pre = os.environ.get("GSPLAT_PREBUILT_DIR", "/kaggle/tmp/gsplat_prebuilt")\n'
           '    if os.path.exists(os.path.join(_pre, "gsplat_cuda.so")):\n'
           '        _C = _import_module_from_library("gsplat_cuda", _pre, True)\n'
           '    else:\n'
           '        from gsplat import csrc as _C\n')
    assert src.count(old) == 1, 'gsplat _backend.py changed, patch needs updating'
    open(BACKEND, 'w').write(src.replace(old, new)); print('patched gsplat loader')

# 2. Restore the cached library from Drive
os.makedirs(PREBUILT, exist_ok=True)
if not os.path.exists(f'{PREBUILT}/gsplat_cuda.so') and os.path.exists(f'{JIT_CACHE}/gsplat_cuda.so'):
    shutil.copy(f'{JIT_CACHE}/gsplat_cuda.so', PREBUILT)

def load_gsplat_cuda():
    t = time.time()
    r = subprocess.run([sys.executable, '-c', 'from gsplat.cuda._backend import _C; print("SO=", _C.__file__ if _C else None)'],
                       capture_output=True, text=True)
    so = re.search(r'SO= (.+)', r.stdout)
    return (so.group(1).strip() if so else None), time.time() - t, r

so, dt, r = load_gsplat_cuda()
if so and so.startswith(PREBUILT):
    print(f'✅ gsplat CUDA loaded from the cache in {dt:.0f}s ({KEY})')
else:
    # 3. No usable cache: the import above compiled it (~15 min). Save the result for next time.
    if not so or so == 'None':
        print(r.stdout[-1500:], r.stderr[-3000:]); raise RuntimeError('gsplat CUDA failed to load or compile')
    print(f'compiled gsplat CUDA in {dt/60:.1f} min -> saving to the Output')
    os.makedirs(JIT_CACHE, exist_ok=True)
    shutil.copy(so, f'{JIT_CACHE}/gsplat_cuda.so'); shutil.copy(so, f'{PREBUILT}/gsplat_cuda.so')
    so2, dt2, _ = load_gsplat_cuda()
    print(f"{'✅' if so2 and so2.startswith(PREBUILT) else '❌'} reload from cache: {dt2:.0f}s ({so2})")

## 3. Research code

Copies of `hybrid_sfm/` in the repo (`r5_eval.py`'s pose math is tested locally: `python hybrid_sfm/tests/test_r5.py`).

In [ ]:
%%writefile /kaggle/tmp/hybrid_sfm/colmap_io.py
"""Minimal COLMAP model I/O (binary and text), numpy only.

Written for the research track so pose evaluation and subsetting don't depend on a particular pycolmap
version (gsplat's examples install a pycolmap *fork* that shadows the official package; VGGT's
demo_colmap.py needs official pycolmap 3.10).

Model = dict with
    cameras: {camera_id: dict(model, width, height, params)}
    images:  {image_id: dict(qvec (w,x,y,z), tvec, camera_id, name, xys (P,2), point3D_ids (P,))}
    points:  {point3D_id: dict(xyz, rgb, error, image_ids, point2D_idxs)}
Poses are world-to-camera (COLMAP / OpenCV convention): x_cam = R(q) @ x_world + t.
"""
import os
import struct

import numpy as np

CAMERA_MODELS = {  # id: (name, number of params)
    0: ("SIMPLE_PINHOLE", 3), 1: ("PINHOLE", 4), 2: ("SIMPLE_RADIAL", 4), 3: ("RADIAL", 5),
    4: ("OPENCV", 8), 5: ("OPENCV_FISHEYE", 8), 6: ("FULL_OPENCV", 12), 7: ("FOV", 5),
    8: ("SIMPLE_RADIAL_FISHEYE", 4), 9: ("RADIAL_FISHEYE", 5), 10: ("THIN_PRISM_FISHEYE", 12),
}
MODEL_ID = {name: i for i, (name, _) in CAMERA_MODELS.items()}


def _read(f, fmt):
    return struct.unpack("<" + fmt, f.read(struct.calcsize("<" + fmt)))


# ------------------------------------------------------------------ binary
def read_cameras_bin(path):
    cams = {}
    with open(path, "rb") as f:
        for _ in range(_read(f, "Q")[0]):
            cid, mid, w, h = _read(f, "iiQQ")
            name, n = CAMERA_MODELS[mid]
            cams[cid] = dict(model=name, width=w, height=h, params=np.array(_read(f, "d" * n)))
    return cams


def read_images_bin(path):
    imgs = {}
    with open(path, "rb") as f:
        for _ in range(_read(f, "Q")[0]):
            iid, = _read(f, "i")
            q = np.array(_read(f, "dddd"))
            t = np.array(_read(f, "ddd"))
            cid, = _read(f, "i")
            name = b""
            while (c := f.read(1)) != b"\x00":
                name += c
            n, = _read(f, "Q")
            raw = np.frombuffer(f.read(24 * n), dtype=np.dtype([("x", "<f8"), ("y", "<f8"), ("id", "<i8")]))
            imgs[iid] = dict(qvec=q, tvec=t, camera_id=cid, name=name.decode(),
                             xys=np.stack([raw["x"], raw["y"]], 1), point3D_ids=raw["id"].copy())
    return imgs


def read_points_bin(path):
    pts = {}
    with open(path, "rb") as f:
        for _ in range(_read(f, "Q")[0]):
            pid, = _read(f, "Q")
            xyz = np.array(_read(f, "ddd"))
            rgb = np.array(_read(f, "BBB"), np.uint8)
            err, n = _read(f, "dQ")
            tr = np.frombuffer(f.read(8 * n), dtype="<i4").reshape(-1, 2)
            pts[pid] = dict(xyz=xyz, rgb=rgb, error=err, image_ids=tr[:, 0].copy(), point2D_idxs=tr[:, 1].copy())
    return pts


def write_model_bin(model, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "cameras.bin"), "wb") as f:
        f.write(struct.pack("<Q", len(model["cameras"])))
        for cid, c in sorted(model["cameras"].items()):
            f.write(struct.pack("<iiQQ", cid, MODEL_ID[c["model"]], int(c["width"]), int(c["height"])))
            f.write(struct.pack("<" + "d" * len(c["params"]), *map(float, c["params"])))
    with open(os.path.join(out_dir, "images.bin"), "wb") as f:
        f.write(struct.pack("<Q", len(model["images"])))
        for iid, im in sorted(model["images"].items()):
            f.write(struct.pack("<i4d3di", iid, *map(float, im["qvec"]), *map(float, im["tvec"]), im["camera_id"]))
            f.write(im["name"].encode() + b"\x00")
            xys, ids = np.asarray(im["xys"], float).reshape(-1, 2), np.asarray(im["point3D_ids"], np.int64)
            f.write(struct.pack("<Q", len(ids)))
            rec = np.empty(len(ids), dtype=np.dtype([("x", "<f8"), ("y", "<f8"), ("id", "<i8")]))
            rec["x"], rec["y"], rec["id"] = xys[:, 0], xys[:, 1], ids
            f.write(rec.tobytes())
    with open(os.path.join(out_dir, "points3D.bin"), "wb") as f:
        f.write(struct.pack("<Q", len(model["points"])))
        for pid, p in sorted(model["points"].items()):
            f.write(struct.pack("<Q3d3Bd", pid, *map(float, p["xyz"]), *map(int, p["rgb"]), float(p["error"])))
            tr = np.stack([p["image_ids"], p["point2D_idxs"]], 1).astype("<i4")
            f.write(struct.pack("<Q", len(tr)) + tr.tobytes())


# ------------------------------------------------------------------ text (read only)
def _data_lines(path):
    with open(path) as f:
        return [l.rstrip("\n") for l in f if not l.startswith("#")]


def read_model_txt(d):
    cams, imgs, pts = {}, {}, {}
    for l in _data_lines(os.path.join(d, "cameras.txt")):
        if not l.strip():
            continue
        v = l.split()
        cams[int(v[0])] = dict(model=v[1], width=int(v[2]), height=int(v[3]), params=np.array(v[4:], float))
    lines = _data_lines(os.path.join(d, "images.txt"))   # pairs: header line, observations line (may be empty)
    i = 0
    while i < len(lines):
        if not lines[i].strip():
            i += 1
            continue
        v = lines[i].split()
        obs = np.array(lines[i + 1].split(), float).reshape(-1, 3) if i + 1 < len(lines) else np.zeros((0, 3))
        imgs[int(v[0])] = dict(qvec=np.array(v[1:5], float), tvec=np.array(v[5:8], float), camera_id=int(v[8]),
                               name=" ".join(v[9:]), xys=obs[:, :2], point3D_ids=obs[:, 2].astype(np.int64))
        i += 2
    for l in _data_lines(os.path.join(d, "points3D.txt")):
        if not l.strip():
            continue
        v = l.split()
        tr = np.array(v[8:], int).reshape(-1, 2)
        pts[int(v[0])] = dict(xyz=np.array(v[1:4], float), rgb=np.array(v[4:7], np.uint8), error=float(v[7]),
                              image_ids=tr[:, 0], point2D_idxs=tr[:, 1])
    return dict(cameras=cams, images=imgs, points=pts)


def find_model_dir(scene_dir):
    """gsplat's rule: sparse/0 if it exists, else sparse."""
    for d in (os.path.join(scene_dir, "sparse", "0"), os.path.join(scene_dir, "sparse")):
        if os.path.exists(os.path.join(d, "images.bin")) or os.path.exists(os.path.join(d, "images.txt")):
            return d
    raise FileNotFoundError(f"no COLMAP model under {scene_dir}/sparse")


def read_model(d):
    if os.path.isdir(os.path.join(d, "sparse")):
        d = find_model_dir(d)
    if os.path.exists(os.path.join(d, "images.bin")):
        return dict(cameras=read_cameras_bin(os.path.join(d, "cameras.bin")),
                    images=read_images_bin(os.path.join(d, "images.bin")),
                    points=read_points_bin(os.path.join(d, "points3D.bin")))
    return read_model_txt(d)


# ------------------------------------------------------------------ helpers
def qvec2rotmat(q):
    w, x, y, z = q / np.linalg.norm(q)
    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y)],
                     [2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x)],
                     [2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)]])


def rotmat2qvec(R):
    K = np.array([[R[0, 0] - R[1, 1] - R[2, 2], 0, 0, 0],
                  [R[1, 0] + R[0, 1], R[1, 1] - R[0, 0] - R[2, 2], 0, 0],
                  [R[2, 0] + R[0, 2], R[2, 1] + R[1, 2], R[2, 2] - R[0, 0] - R[1, 1], 0],
                  [R[2, 1] - R[1, 2], R[0, 2] - R[2, 0], R[1, 0] - R[0, 1], R[0, 0] + R[1, 1] + R[2, 2]]]) / 3
    w, V = np.linalg.eigh(K)
    q = V[[3, 0, 1, 2], np.argmax(w)]
    return q if q[0] >= 0 else -q


def poses_by_name(model):
    """{image name: (R (3,3), t (3,))}, world-to-camera."""
    return {im["name"]: (qvec2rotmat(im["qvec"]), np.asarray(im["tvec"], float)) for im in model["images"].values()}


def subset_model(model, keep_names, min_track=2):
    """Keep only images in keep_names; keep 3D points still seen by >= min_track of them.

    gsplat looks every image in the model up in images/ and every track entry up in the model's images,
    so a subset needs both trimmed. Dropped points are set to -1 in the images' observations.
    """
    keep_names = set(keep_names)
    images = {i: dict(im) for i, im in model["images"].items() if im["name"] in keep_names}
    missing = keep_names - {im["name"] for im in images.values()}
    if missing:
        raise KeyError(f"{len(missing)} names not in the model, e.g. {sorted(missing)[:3]}")
    kept_ids = np.array(sorted(images))
    points = {}
    for pid, p in model["points"].items():
        m = np.isin(p["image_ids"], kept_ids)
        if m.sum() >= min_track:
            points[pid] = dict(p, image_ids=p["image_ids"][m], point2D_idxs=p["point2D_idxs"][m])
    for im in images.values():
        ids = np.asarray(im["point3D_ids"], np.int64).copy()
        ids[(ids >= 0) & ~np.isin(ids, np.fromiter(points.keys(), np.int64, len(points)))] = -1
        im["point3D_ids"] = ids
    cams = {c: model["cameras"][c] for c in {im["camera_id"] for im in images.values()}}
    return dict(cameras=cams, images=images, points=points)

In [ ]:
%%writefile /kaggle/tmp/hybrid_sfm/pose_metrics.py
"""Camera pose accuracy against a reference, numpy only.

Metrics (the usual ones in the VGGT / VGGSfM / MASt3R-SfM papers):
- RRA@τ / RTA@τ: share of image pairs whose relative rotation / relative translation direction error is < τ°.
  Translation direction is sign-ambiguous for some methods, so its error is min(e, 180 - e).
- AUC@T: mean over 30 evenly spaced τ in (0, T] of the share of pairs with max(rotation error, translation error) < τ.
  AUC@30 uses τ = 1..30 (the usual definition). AUC@3 and AUC@5 resolve the sub-degree errors that matter for
  3DGS: in R1, VGGT scored AUC@30 = 0.995 and RRA@5 = 1.0 and still lost 4.2 dB of PSNR.
- focal_error: median relative error of f / image width against the reference (R2).
- ATE: RMSE of camera centres after a Sim(3) (Umeyama) alignment to the reference, divided by the
  reference's scene scale (median distance of reference centres to their mean), in percent.
Images the method failed to register count as 180° error for every pair they are in, so methods that
drop images are not rewarded for it; the share of registered images is reported too.
"""
import numpy as np


def _angle(R):
    return np.degrees(np.arccos(np.clip((np.trace(R) - 1) / 2, -1.0, 1.0)))


def _vec_angle(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < 1e-12 or nb < 1e-12:
        return 180.0
    return np.degrees(np.arccos(np.clip(a @ b / (na * nb), -1.0, 1.0)))


def umeyama(src, dst):
    """s, R, t minimising ||s R src + t - dst||^2 (src, dst: (N, 3))."""
    mu_s, mu_d = src.mean(0), dst.mean(0)
    xs, xd = src - mu_s, dst - mu_d
    U, S, Vt = np.linalg.svd(xd.T @ xs / len(src))
    D = np.eye(3)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        D[2, 2] = -1
    R = U @ D @ Vt
    var_s = (xs ** 2).sum() / len(src)
    s = np.trace(np.diag(S) @ D) / var_s if var_s > 0 else 1.0
    return s, R, mu_d - s * R @ mu_s


def centre(R, t):
    return -R.T @ t


def pose_errors(pred, ref, names=None):
    """pred, ref: {name: (R, t)} world-to-camera. names: the image set to score (default: all of ref)."""
    names = sorted(ref if names is None else names)
    reg = [n for n in names if n in pred]
    out = dict(n_images=len(names), n_registered=len(reg), registered=len(reg) / max(len(names), 1))

    rot_err, tr_err = [], []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]
            if a not in pred or b not in pred:
                rot_err.append(180.0); tr_err.append(180.0)
                continue
            (Ra, ta), (Rb, tb) = ref[a], ref[b]
            (Pa, pa), (Pb, pb) = pred[a], pred[b]
            R_ref, R_pred = Rb @ Ra.T, Pb @ Pa.T                       # camera a -> camera b
            t_ref, t_pred = tb - R_ref @ ta, pb - R_pred @ pa
            rot_err.append(_angle(R_pred.T @ R_ref))
            e = _vec_angle(t_pred, t_ref)
            tr_err.append(min(e, 180.0 - e))
    rot_err, tr_err = np.array(rot_err), np.array(tr_err)
    worst = np.maximum(rot_err, tr_err)
    for tau in (1, 5, 15):
        out[f"RRA@{tau}"] = float((rot_err < tau).mean()) if len(worst) else float("nan")
        out[f"RTA@{tau}"] = float((tr_err < tau).mean()) if len(worst) else float("nan")
    for T in (3, 5, 30):
        out[f"AUC@{T}"] = float(np.mean([(worst < t).mean() for t in np.arange(1, 31) * T / 30])) if len(worst) else float("nan")
    out["rot_err_median"] = float(np.median(rot_err)) if len(rot_err) else float("nan")
    out["tr_err_median"] = float(np.median(tr_err)) if len(tr_err) else float("nan")

    if len(reg) >= 3:
        C_ref = np.array([centre(*ref[n]) for n in reg])
        C_pred = np.array([centre(*pred[n]) for n in reg])
        s, R, t = umeyama(C_pred, C_ref)
        res = (s * C_pred @ R.T + t) - C_ref
        scale = np.median(np.linalg.norm(C_ref - C_ref.mean(0), axis=1))
        out["ATE_%"] = float(100 * np.sqrt((res ** 2).sum(1).mean()) / max(scale, 1e-12))
    else:
        out["ATE_%"] = float("nan")
    return out


def _f_over_w(cam):
    p, m = np.asarray(cam["params"], float), cam["model"]
    f = (p[0] + p[1]) / 2 if m in ("PINHOLE", "OPENCV", "OPENCV_FISHEYE", "FULL_OPENCV", "THIN_PRISM_FISHEYE") else p[0]
    return f / cam["width"]


def focal_error(pred_model, ref_model, names=None):
    """Median |f/W - f_ref/W_ref| / (f_ref/W_ref) over registered images (models as in colmap_io)."""
    ref = {im["name"]: _f_over_w(ref_model["cameras"][im["camera_id"]]) for im in ref_model["images"].values()}
    pred = {im["name"]: _f_over_w(pred_model["cameras"][im["camera_id"]]) for im in pred_model["images"].values()}
    names = [n for n in (names if names is not None else ref) if n in pred and n in ref]
    if not names:
        return float("nan")
    return float(np.median([abs(pred[n] - ref[n]) / ref[n] for n in names]))

In [ ]:
%%writefile /kaggle/tmp/hybrid_sfm/chunks.py
"""R2 chunk sampler: deterministic image subsets of one scene with varied difficulty.

A chunk is what the router decides on. To get labels where each reconstructor sometimes wins and sometimes
fails, chunks vary in size and in baseline:
- window: n consecutive images (images are sorted by name, i.e. capture order): narrow baseline, partial view;
- stride: n images evenly spaced over the whole capture: wide baseline, sparse 360° coverage (R1's bicycle
  setting, where COLMAP registered 11 of 32);
- random: n images drawn at random: irregular overlap.
15 chunks per scene by default (6 window, 5 stride, 4 random); sizes stay <= 40 so one VGGT pass fits a T4 (R1).
"""
import hashlib

import numpy as np

DEFAULT_SPECS = ([("window", 24, k) for k in range(4)] + [("window", 40, k) for k in range(2)]
                 + [("stride", n, 0) for n in (12, 16, 24, 32, 40)]
                 + [("random", n, k) for k, n in enumerate((16, 24, 32, 40))])


def _seed(*parts):
    return int(hashlib.sha256("/".join(map(str, parts)).encode()).hexdigest()[:8], 16)


def make_chunks(scene, names, specs=DEFAULT_SPECS):
    """[{id, scene, kind, n, k, names}] for one scene. Same input -> same chunks, on any machine."""
    names = sorted(names)
    N = len(names)
    out = []
    for kind, n, k in specs:
        n = min(n, N)
        if kind == "window":
            n_windows = sum(1 for s in specs if s[0] == "window" and s[1] == n)
            start = int(round(k * (N - n) / max(n_windows, 1))) if N > n else 0
            idx = np.arange(start, start + n)
        elif kind == "stride":
            idx = np.unique(np.linspace(0, N - 1, n).round().astype(int))
        elif kind == "random":
            idx = np.sort(np.random.default_rng(_seed(scene, kind, n, k)).choice(N, n, replace=False))
        else:
            raise ValueError(kind)
        out.append(dict(id=f"{scene}_{kind}{n}_{k}", scene=scene, kind=kind, n=int(len(idx)), k=k,
                        names=[names[i] for i in idx]))
    return out

In [ ]:
%%writefile /kaggle/tmp/hybrid_sfm/r5_eval.py
"""R5.1: independent scoring of a 3DGS twin trained on a method's poses.

R2's label renders the held-out views at the method's *own* estimated poses. That is deployable, but it shares a
measurement with the R3b probe, and it never checks the twin against the true cameras. This script scores a trained
twin the way pose-free novel-view-synthesis papers do:

  1. aligned:  a Sim(3) from the reference cameras to the twin's own *training* cameras (Umeyama on camera centres,
               in gsplat's normalized training frame), applied to the held-out reference cameras; rendered with the
               reference intrinsics (the real camera), compared with the real photo;
  2. refined:  starting from (1), each held-out pose is refined by test-time optimisation (Gaussians frozen, Adam on
               an L1 photometric loss), the standard protocol of NoPe-NeRF, CF-3DGS and others. This removes the
               alignment error and measures what the twin can render.

Runs in gsplat's examples/ folder (imports its COLMAP Parser so the split and the normalisation match training):
    python r5_eval.py --data_dir <method scene> --ref_model <ref sparse dir> --ckpt <ckpt.pt> --out <json>
"""
import argparse
import json
import os
import sys

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
try:
    from hybrid_sfm import colmap_io as cio, pose_metrics as pm          # in the repo
except ImportError:                                                      # copied next to colmap_io.py (notebook)
    import colmap_io as cio
    import pose_metrics as pm


# ------------------------------------------------------------------ pose math (numpy, tested locally)
def view_in_frame(R_c, t_c, s, R, t):
    """Reference world->camera (R_c, t_c), and a Sim(3) x_frame = s R x_ref + t.
    Returns world->camera in the frame, with camera coordinates scaled by s (projection unchanged)."""
    R2 = R_c @ R.T
    return R2, s * t_c - R2 @ t


def intrinsics(cam, width):
    """Pinhole K of a COLMAP camera, scaled to an image `width` pixels wide (distortion terms are ignored)."""
    p, m = np.asarray(cam["params"], float), cam["model"]
    if m in ("SIMPLE_PINHOLE", "SIMPLE_RADIAL", "RADIAL", "SIMPLE_RADIAL_FISHEYE", "RADIAL_FISHEYE"):
        fx = fy = p[0]; cx, cy = p[1], p[2]
    else:                                                             # PINHOLE, OPENCV, FULL_OPENCV, ...
        fx, fy, cx, cy = p[:4]
    k = width / cam["width"]
    return np.array([[fx * k, 0, cx * k], [0, fy * k, cy * k], [0, 0, 1.0]])


def align_ref_to_frame(ref_poses, frame_c2w, names):
    """Sim(3) taking reference camera centres onto the twin's training camera centres (frame_c2w: {name: 4x4})."""
    common = [n for n in names if n in ref_poses and n in frame_c2w]
    if len(common) < 3:
        raise ValueError(f"only {len(common)} shared training cameras")
    A = np.array([pm.centre(*ref_poses[n]) for n in common])
    B = np.array([frame_c2w[n][:3, 3] for n in common])
    return pm.umeyama(A, B)


# ------------------------------------------------------------------ rendering (GPU)
def _so3_exp(w):
    import torch
    th = torch.sqrt((w * w).sum() + 1e-12)                  # smooth at w = 0 (the optimisation starts there)
    k = w / th
    Kx = torch.zeros(3, 3, device=w.device, dtype=w.dtype)
    Kx[0, 1], Kx[0, 2], Kx[1, 0], Kx[1, 2], Kx[2, 0], Kx[2, 1] = -k[2], k[1], k[2], -k[0], -k[1], k[0]
    return torch.eye(3, device=w.device, dtype=w.dtype) + torch.sin(th) * Kx + (1 - torch.cos(th)) * Kx @ Kx


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True, help="the method's scene folder the twin was trained on")
    ap.add_argument("--ref_model", required=True, help="reference COLMAP model of the same chunk")
    ap.add_argument("--ckpt", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--test_every", type=int, default=8)
    ap.add_argument("--tto_iters", type=int, default=300)
    ap.add_argument("--tto_lr", type=float, default=2e-3)
    a = ap.parse_args()

    import imageio.v2 as imageio
    import torch
    from gsplat.rendering import rasterization
    from datasets.colmap import Parser                                  # gsplat examples/datasets
    from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure

    dev = "cuda"
    parser = Parser(data_dir=a.data_dir, factor=1, normalize=True, test_every=a.test_every)
    names = list(parser.image_names)
    idx = np.arange(len(names))
    test_i, train_i = idx[idx % a.test_every == 0], idx[idx % a.test_every != 0]
    c2w = {n: parser.camtoworlds[i] for i, n in enumerate(names)}

    ref = cio.read_model(a.ref_model)
    ref_poses = cio.poses_by_name(ref)
    ref_cam = {im["name"]: ref["cameras"][im["camera_id"]] for im in ref["images"].values()}
    s, R, t = align_ref_to_frame(ref_poses, c2w, [names[i] for i in train_i])

    sp = torch.load(a.ckpt, map_location=dev)["splats"]
    means, quats = sp["means"], torch.nn.functional.normalize(sp["quats"], dim=-1)
    scales, opac = torch.exp(sp["scales"]), torch.sigmoid(sp["opacities"])
    colors = torch.cat([sp["sh0"], sp["shN"]], 1)
    sh_degree = int(round(np.sqrt(colors.shape[1]))) - 1

    def render(viewmat, K, W, H):
        out, _, _ = rasterization(means, quats, scales, opac, colors, viewmat[None], K[None], W, H, sh_degree=sh_degree)
        return out[0].clamp(0, 1)

    psnr = PeakSignalNoiseRatio(data_range=1.0).to(dev)
    ssim = StructuralSimilarityIndexMeasure(data_range=1.0).to(dev)
    score = lambda img, gt: (float(psnr(img.permute(2, 0, 1)[None], gt.permute(2, 0, 1)[None])),
                             float(ssim(img.permute(2, 0, 1)[None], gt.permute(2, 0, 1)[None])))
    per = []
    for i in test_i:
        n = names[i]
        if n not in ref_poses:
            continue
        gt = torch.from_numpy(imageio.imread(parser.image_paths[i])[..., :3].astype(np.float32) / 255).to(dev)
        H, W = gt.shape[:2]
        R2, t2 = view_in_frame(*ref_poses[n], s, R, t)
        V0 = torch.eye(4, device=dev)
        V0[:3, :3], V0[:3, 3] = torch.tensor(R2, device=dev), torch.tensor(t2, device=dev)
        K = torch.tensor(intrinsics(ref_cam[n], W), device=dev, dtype=torch.float32)
        with torch.no_grad():
            p0, s0 = score(render(V0, K, W, H), gt)
        own = torch.linalg.inv(torch.tensor(c2w[n], device=dev, dtype=torch.float32))           # the method's own pose
        Kown = torch.tensor(parser.Ks_dict[parser.camera_ids[i]], device=dev, dtype=torch.float32)
        with torch.no_grad():
            p_own, _ = score(render(own, Kown, W, H), gt)
        w = torch.zeros(3, device=dev, requires_grad=True)
        u = torch.zeros(3, device=dev, requires_grad=True)
        opt = torch.optim.Adam([w, u], lr=a.tto_lr)
        for _ in range(a.tto_iters):
            D = torch.eye(4, device=dev)
            D = D.clone()
            D[:3, :3] = _so3_exp(w)
            D[:3, 3] = u
            img = render(D @ V0, K, W, H)
            loss = (img - gt).abs().mean()
            opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            D = torch.eye(4, device=dev); D[:3, :3] = _so3_exp(w); D[:3, 3] = u
            p1, s1 = score(render(D @ V0, K, W, H), gt)
        per.append(dict(name=n, psnr_aligned=p0, ssim_aligned=s0, psnr_refined=p1, ssim_refined=s1, psnr_own_pose=p_own,
                        tto_rot_deg=float(np.degrees(torch.linalg.norm(w).item())), tto_trans=float(torch.linalg.norm(u).item())))
    res = dict(n_test=len(per), sim3_scale=float(s),
               **{k: float(np.mean([p[k] for p in per])) for k in ("psnr_aligned", "ssim_aligned", "psnr_refined",
                                                                  "ssim_refined", "psnr_own_pose")} if per else {},
               per_image=per)
    json.dump(res, open(a.out, "w"), indent=1)
    print(json.dumps({k: v for k, v in res.items() if k != "per_image"}))


if __name__ == "__main__":
    main()

In [ ]:
open(f'{WORK}/hybrid_sfm/__init__.py', 'a').close()
sys.path.insert(0, WORK)
from hybrid_sfm import colmap_io as cio, chunks
print('hybrid_sfm ready')

## 4. Data: Mip-NeRF 360 scenes (the resolution R2 used)

In [ ]:
from remotezip import RemoteZip
import imageio.v2 as imageio, numpy as np
URL = 'http://storage.googleapis.com/gresearch/refraw360/360_v2.zip'

def get_scene(scene):
    # Downloads only the files that are missing or incomplete, so an interrupted download is finished next time.
    data, want = f'{WORK}/data/360_v2/{scene}', ('sparse/', 'poses_bounds.npy', RES[scene] + '/')
    if not os.path.exists(f'{data}/.complete_{RES[scene]}'):
        t = time.time(); n = 0
        with RemoteZip(URL) as z:
            for info in z.infolist():
                m = info.filename
                if not (m.startswith(f'{scene}/') and not m.endswith('/') and m.split('/', 1)[1].startswith(want)):
                    continue
                dst = f'{data}/' + m.split('/', 1)[1]
                if os.path.exists(dst) and os.path.getsize(dst) == info.file_size:
                    continue
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                with z.open(m) as src_, open(dst + '.part', 'wb') as f:
                    f.write(src_.read())
                os.replace(dst + '.part', dst); n += 1
        open(f'{data}/.complete_{RES[scene]}', 'w').write('ok')
        print(f'{scene}: downloaded {n} files in {time.time()-t:.0f}s')
    return data

## 5. Chunks (deterministic: the same as R2's)

In [ ]:
SCENE_DATA, SCENE_REF, CHUNKS = {}, {}, []
for scene in SCENES:
    data = get_scene(scene)
    ref = cio.read_model(data)                                     # official model: the reference
    in_model = {im['name'] for im in ref['images'].values()}
    names = [n for n in os.listdir(f'{data}/{RES[scene]}') if n in in_model]
    SCENE_DATA[scene], SCENE_REF[scene] = data, ref
    cs = chunks.make_chunks(scene, names)                          # deterministic: the same chunks as R2
    CHUNKS += cs
    print(f'{scene:>8}: {len(names)} images -> {len(cs)} chunks')
missing = [c['id'] for c in CHUNKS if not os.path.isdir(f"{MODELS}/{c['id']}")]
assert not missing, f'chunks without R2 models: {missing[:5]}'

DONE = set()
if os.path.exists(SCORES):
    for line in open(SCORES):
        if line.strip():
            r = json.loads(line); DONE.add((r['chunk'], r['method']))
todo = [c for c in CHUNKS if not all((c['id'], m) in DONE for m in METHODS)]
print(f'{len(CHUNKS)} chunks, {len(CHUNKS) - len(todo)} already scored, {len(todo)} to go')

## 6. Train and score

For every chunk: the official poses and every usable method get R2's 3k-step recipe; the checkpoint is scored three ways (own pose, aligned, refined).

In [ ]:
GS_ENV = {**os.environ, 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}
EVAL_ENV = {**GS_ENV, 'PYTHONPATH': f'{REPO}/examples:{WORK}'}      # gsplat's datasets.colmap + hybrid_sfm
help_txt = subprocess.run([sys.executable, 'simple_trainer.py', 'mcmc', '--help'], capture_output=True, text=True,
                          cwd=f'{REPO}/examples').stdout
VIDEO_FLAG = ['--disable_video'] if re.search(r'disable.video', help_txt) else []      # F-R1.4

def save_log(name, text):
    open(f'{OUT}/logs/{name}.txt', 'w').write(text[-20000:])

def r2_model_dir(c, m):
    d = f"{MODELS}/{c['id']}/{m}"
    info = json.load(open(f'{d}/info.json')) if os.path.exists(f'{d}/info.json') else dict(status='missing')
    for s in (f'{d}/sparse/0', f'{d}/sparse'):
        if os.path.exists(f'{s}/images.bin'):
            return s, info
    return None, info

def prepare(c, m, src=None):
    d = f"{R5}/{c['id']}/{m}"
    os.makedirs(f'{d}/images', exist_ok=True)
    for n in c['names']:
        if not os.path.exists(f'{d}/images/{n}'):
            os.symlink(f"{SCENE_DATA[c['scene']]}/{RES[c['scene']]}/{n}", f'{d}/images/{n}')
    if not os.path.exists(f'{d}/sparse/0/images.bin'):
        if m == 'ref':
            cio.write_model_bin(cio.subset_model(SCENE_REF[c['scene']], c['names']), f'{d}/sparse/0')
        else:
            shutil.copytree(src, f'{d}/sparse/0', dirs_exist_ok=True)
    return d

def train_and_score(data_dir, ref_dir, tag):
    # R2's recipe (3k steps, checkpoint kept), then r5_eval.py on the checkpoint. Run folder deleted afterwards.
    rd = f'{WORK}/gs_runs/{tag}'
    shutil.rmtree(rd, ignore_errors=True)
    cmd = [sys.executable, 'simple_trainer.py', 'mcmc', '--strategy.cap-max', str(CAP), '--data_dir', data_dir,
           '--data_factor', '1', '--result_dir', rd, '--max_steps', str(STEPS), '--eval_steps', str(STEPS),
           '--save_steps', str(STEPS), '--disable_viewer', '--lpips_net', 'vgg', *VIDEO_FLAG]
    t = time.time()
    r = subprocess.run(cmd, cwd=f'{REPO}/examples', capture_output=True, text=True, env=GS_ENV, timeout=3600)
    vals, ckpts = glob.glob(f'{rd}/stats/val_step*.json'), glob.glob(f'{rd}/ckpts/*.pt')
    if r.returncode != 0 or not vals or not ckpts:
        save_log(f'train_{tag}', r.stdout + '\n----- stderr -----\n' + r.stderr)
        raise RuntimeError('3DGS failed: ' + ([l for l in r.stderr.strip().splitlines() if l.strip()] or ['?'])[-1][:200])
    gs = json.load(open(vals[-1])); train_s = time.time() - t
    t = time.time()
    e = subprocess.run([sys.executable, f'{WORK}/hybrid_sfm/r5_eval.py', '--data_dir', data_dir, '--ref_model', ref_dir,
                        '--ckpt', ckpts[-1], '--out', f'{rd}/r5.json', '--tto_iters', str(TTO_ITERS)],
                       cwd=f'{REPO}/examples', capture_output=True, text=True, env=EVAL_ENV, timeout=1800)
    if e.returncode != 0 or not os.path.exists(f'{rd}/r5.json'):
        save_log(f'eval_{tag}', e.stdout + '\n----- stderr -----\n' + e.stderr)
        raise RuntimeError('r5_eval failed: ' + ([l for l in e.stderr.strip().splitlines() if l.strip()] or ['?'])[-1][:200])
    ind = json.load(open(f'{rd}/r5.json'))
    shutil.rmtree(rd, ignore_errors=True)
    return dict(psnr=gs['psnr'], ssim=gs['ssim'], lpips=gs['lpips'], train_s=round(train_s, 1)), ind, round(time.time() - t, 1)

def score_chunk(c):
    ref_dir = f"{prepare(c, 'ref')}/sparse/0"
    for m in METHODS:
        if (c['id'], m) in DONE:
            continue
        rec = dict(chunk=c['id'], scene=c['scene'], kind=c['kind'], n=c['n'], method=m, gs=None, indep=None)
        src, info = (ref_dir, dict(status='ok')) if m == 'ref' else r2_model_dir(c, m)
        if src is None or info.get('status') != 'ok':
            rec['status'] = 'recon_failed'
        elif m != 'ref' and len(cio.read_images_bin(f'{src}/images.bin')) / c['n'] < MIN_COVERAGE:
            rec['status'] = 'low_coverage'
        else:
            try:
                rec['gs'], rec['indep'], rec['eval_s'] = train_and_score(prepare(c, m, src), ref_dir, f"{c['id']}_{m}")
                rec['status'] = 'ok'
            except Exception as e:
                rec['status'] = 'failed'; rec['error'] = str(e)[:300]
        with open(SCORES, 'a') as f:
            f.write(json.dumps(rec, default=float) + '\n')
        DONE.add((c['id'], m))
        g, ind = rec['gs'], rec['indep'] or {}
        print(f"  {m:<9} {rec['status']:<13}" + (f" own {g['psnr']:.2f} | aligned {ind.get('psnr_aligned', float('nan')):.2f}"
              f" | refined {ind.get('psnr_refined', float('nan')):.2f} ({rec['eval_s']:.0f} s eval)" if g else ''))
    shutil.rmtree(f"{R5}/{c['id']}", ignore_errors=True)

## 7. Run (resumable: scored chunks are skipped)

In [ ]:
for i, c in enumerate(todo):
    if (time.time() - T_START) / 3600 > BUDGET_H:
        print(f'⏸ time budget of {BUDGET_H} h reached: {len(todo) - i} chunks left for the next run'); break
    t = time.time()
    print(f"[{i + 1}/{len(todo)}] {c['id']} ({c['n']} images, {c['kind']})")
    try:
        score_chunk(c)
    except Exception as e:                                       # one bad chunk must not stop the run
        print(f"  ❌ {c['id']}: {repr(e)[:300]}")
    print(f'  {(time.time() - t) / 60:.1f} min | {(time.time() - T_START) / 3600:.2f} h elapsed')

## 8. Summary

In [ ]:
from collections import Counter
recs = [json.loads(l) for l in open(SCORES) if l.strip()]
print(len(recs), 'records:', dict(Counter(r['status'] for r in recs)))
ok = [r for r in recs if r['status'] == 'ok' and r['indep'] and 'psnr_refined' in r['indep']]
for m in METHODS:
    rs = [r for r in ok if r['method'] == m]
    if rs:
        f = lambda k: sum(r['indep'][k] for r in rs) / len(rs)
        print(f"{m:<9} n={len(rs):>3}  own-pose {sum(r['gs']['psnr'] for r in rs) / len(rs):.2f}  aligned {f('psnr_aligned'):.2f}  refined {f('psnr_refined'):.2f}")
print(subprocess.run(['du', '-sh', OUT], capture_output=True, text=True).stdout)

## ✅ Checklist
- [ ] If the log ends with `⏸ time budget ... chunks left`: run again with this run's Output as an Input
- [ ] Download the `r5` folder (Output) into `D:\project\3D` and send the path in the chat